# Premier League 2024/25 — Season Analysis

An analysis of the 2024/25 English Premier League season using match-level data,
covering final standings, attacking and defensive performance, home advantage,
and how the title race developed over time.

**Data source:** [football-data.co.uk](https://www.football-data.co.uk/englandm.php) — one row per match (380 matches, 20 teams).

**Key columns used:**
- `Date` — match date
- `HomeTeam`, `AwayTeam` — the two teams
- `FTHG`, `FTAG` — full-time home goals, full-time away goals
- `FTR` — full-time result (H = home win, A = away win, D = draw)

In [1]:
import pandas as pd
df = pd.read_csv('E0.csv')
print(f"Loaded {df.shape[0]} matches with {df.shape[1]} columns")
df.head()

Loaded 380 matches with 120 columns


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,16/08/2024,20:00,Man United,Fulham,1,0,H,0,0,...,1.86,2.07,1.83,2.11,1.88,2.11,1.82,2.05,1.90,2.08
1,E0,17/08/2024,12:30,Ipswich,Liverpool,0,2,A,0,0,...,2.05,1.88,2.04,1.90,2.20,2.00,1.99,1.88,2.04,1.93
2,E0,17/08/2024,15:00,Arsenal,Wolves,2,0,H,1,0,...,2.02,1.91,2.00,1.90,2.05,1.93,1.99,1.87,2.02,1.96
3,E0,17/08/2024,15:00,Everton,Brighton,0,3,A,0,1,...,1.87,2.06,1.86,2.07,1.92,2.10,1.83,2.04,1.88,2.11
4,E0,17/08/2024,15:00,Newcastle,Southampton,1,0,H,1,0,...,1.87,2.06,1.88,2.06,1.89,2.10,1.82,2.05,1.89,2.10


In [2]:
print('Previewing only the columns relevant to the analysis')

df[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']].head()

Previewing only the columns relevant to the analysis


,Date,HomeTeam,AwayTeam,FTHG,FTAG
0,16/08/2024,Man United,Fulham,1,0
1,17/08/2024,Ipswich,Liverpool,0,2
2,17/08/2024,Arsenal,Wolves,2,0
3,17/08/2024,Everton,Brighton,0,3
4,17/08/2024,Newcastle,Southampton,1,0


In [3]:
print('Created the home view prespective for the teams')
home_view = df[['Date', 'HomeTeam', 'FTHG', 'FTAG']].copy()

home_view = home_view.rename(columns={
    'Date': 'date',
    'HomeTeam': 'team',
    'FTHG': 'goals_for',
    'FTAG': 'goals_against',
})

home_view.head()

Created the home view prespective for the teams


,date,team,goals_for,goals_against
0,16/08/2024,Man United,1,0
1,17/08/2024,Ipswich,0,2
2,17/08/2024,Arsenal,2,0
3,17/08/2024,Everton,0,3
4,17/08/2024,Newcastle,1,0


In [4]:
print('Added the venue column')
home_view['venue'] = 'home'
home_view.head()

Added the venue column


,date,team,goals_for,goals_against,venue
0,16/08/2024,Man United,1,0,home
1,17/08/2024,Ipswich,0,2,home
2,17/08/2024,Arsenal,2,0,home
3,17/08/2024,Everton,0,3,home
4,17/08/2024,Newcastle,1,0,home


In [5]:
print('Created the away view prespective for the teams')
away_view = df[['Date', 'AwayTeam', 'FTAG', 'FTHG']].copy()

away_view = away_view.rename(columns={
    'Date': 'date',
    'AwayTeam': 'team',
    'FTAG': 'goals_for',
    'FTHG': 'goals_against',
})

away_view.head()

Created the away view prespective for the teams


,date,team,goals_for,goals_against
0,16/08/2024,Fulham,0,1
1,17/08/2024,Liverpool,2,0
2,17/08/2024,Wolves,0,2
3,17/08/2024,Brighton,3,0
4,17/08/2024,Southampton,0,1


In [6]:
print('Added the venue column')
away_view['venue'] = 'away'

away_view.head()

Added the venue column


,date,team,goals_for,goals_against,venue
0,16/08/2024,Fulham,0,1,away
1,17/08/2024,Liverpool,2,0,away
2,17/08/2024,Wolves,0,2,away
3,17/08/2024,Brighton,3,0,away
4,17/08/2024,Southampton,0,1,away


In [7]:
print('Stacked the home_view and away_view into one Dataframe called team_matches')
team_matches = pd.concat([home_view, away_view], ignore_index=True)

print(f"Rows: {team_matches.shape[0]}")
team_matches.head()

Stacked the home_view and away_view into one Dataframe called team_matches
Rows: 760


,date,team,goals_for,goals_against,venue
0,16/08/2024,Man United,1,0,home
1,17/08/2024,Ipswich,0,2,home
2,17/08/2024,Arsenal,2,0,home
3,17/08/2024,Everton,0,3,home
4,17/08/2024,Newcastle,1,0,home


In [8]:
print('Sanity check for the total games per team')
team_matches['team'].value_counts()

Sanity check for the total games per team


team
Man United        38
Ipswich           38
Arsenal           38
Everton           38
Newcastle         38
Nott'm Forest     38
West Ham          38
Brentford         38
Chelsea           38
Leicester         38
Brighton          38
Crystal Palace    38
Fulham            38
Man City          38
Southampton       38
Tottenham         38
Aston Villa       38
Bournemouth       38
Wolves            38
Liverpool         38
Name: count, dtype: int64

## Deriving result and points

With the data in long format, each row's result can be determined purely from `goals_for` vs `goals_against`:
- scored more than conceded → **win** (3 points)
- equal → **draw** (1 point)
- scored fewer → **loss** (0 points)

In [9]:
import numpy as np

conditions = [
    team_matches['goals_for'] > team_matches['goals_against'],
    team_matches['goals_for'] == team_matches['goals_against'],
]
choices = ['win', 'draw']

team_matches['result'] = np.select(conditions, choices, default='loss')

team_matches.head(10)

,date,team,goals_for,goals_against,venue,result
0,16/08/2024,Man United,1,0,home,win
1,17/08/2024,Ipswich,0,2,home,loss
2,17/08/2024,Arsenal,2,0,home,win
3,17/08/2024,Everton,0,3,home,loss
4,17/08/2024,Newcastle,1,0,home,win
5,17/08/2024,Nott'm Forest,1,1,home,draw
6,17/08/2024,West Ham,1,2,home,loss
7,18/08/2024,Brentford,2,1,home,win
8,18/08/2024,Chelsea,0,2,home,loss
9,19/08/2024,Leicester,1,1,home,draw


In [10]:
points_map = {'win': 3, 'loss': 0, 'draw': 1}

team_matches['points'] = team_matches['result'].map(points_map)

team_matches.head()

,date,team,goals_for,goals_against,venue,result,points
0,16/08/2024,Man United,1,0,home,win,3
1,17/08/2024,Ipswich,0,2,home,loss,0
2,17/08/2024,Arsenal,2,0,home,win,3
3,17/08/2024,Everton,0,3,home,loss,0
4,17/08/2024,Newcastle,1,0,home,win,3


## Final league table

With the long-format data in place, the entire standings table comes from a single `groupby('team')`.

In [11]:
standings = team_matches.groupby('team').agg(
    played = ('result', 'count'),
    wins = ('result', lambda x: (x == 'win').sum()),
    draws = ('result', lambda x: (x == 'draw').sum()),
    losses = ('result', lambda x: (x == 'loss').sum()),
    goals_for = ('goals_for', 'sum'),
    goals_against = ('goals_against', 'sum'),
    points = ('points', 'sum'),
)

standings['goal_diff'] = standings['goals_for'] - standings['goals_against']

standings = standings.sort_values(['points', 'goal_diff'], ascending=False)

standings

,played,wins,draws,losses,goals_for,goals_against,points,goal_diff
team,,,,,,,,
Liverpool,38,25,9,4,86,41,84,45
Arsenal,38,20,14,4,69,34,74,35
Man City,38,21,8,9,72,44,71,28
Chelsea,38,20,9,9,64,43,69,21
Newcastle,38,20,6,12,68,47,66,21
Aston Villa,38,19,9,10,58,51,66,7
Nott'm Forest,38,19,8,11,58,46,65,12
Brighton,38,16,13,9,66,59,61,7
Bournemouth,38,15,11,12,58,46,56,12


## Home advantage

Across the whole league, home teams won more often than away teams. But is that uniform, or do
some teams rely on home form far more than others?

Grouping by both `team` and `venue` splits each team's season into their home and away records.

In [12]:
venue_points = team_matches.groupby(['team', 'venue'])['points'].sum().unstack()
venue_points

venue_points['home_advantage'] = venue_points['home'] - venue_points['away']
venue_points = venue_points.sort_values('home_advantage', ascending=False)
venue_points

venue,away,home,home_advantage
team,,,
Aston Villa,26,40,14
Chelsea,28,41,13
Man City,29,42,13
Newcastle,28,38,10
Liverpool,38,46,8
Man United,18,24,6
Brentford,25,31,6
Leicester,10,15,5
Tottenham,17,21,4


In [13]:
team_matches.dtypes

date               str
team               str
goals_for        int64
goals_against    int64
venue              str
result             str
points           int64
dtype: object

In [14]:
team_matches['date'] = pd.to_datetime(team_matches['date'], format='%d/%m/%Y')
team_matches.dtypes

date             datetime64[us]
team                        str
goals_for                 int64
goals_against             int64
venue                       str
result                      str
points                    int64
dtype: object

In [15]:
team_matches = team_matches.sort_values('date')

team_matches['cumulative_points'] = team_matches.groupby('team')['points'].cumsum()

team_matches[team_matches['team'] == 'Liverpool'].head(38)

,date,team,goals_for,goals_against,venue,result,points,cumulative_points
381,2024-08-17,Liverpool,2,0,away,win,3,3
19,2024-08-25,Liverpool,2,0,home,win,3,6
409,2024-09-01,Liverpool,3,0,away,win,3,9
34,2024-09-14,Liverpool,0,1,home,loss,0,9
44,2024-09-21,Liverpool,3,0,home,win,3,12
436,2024-09-28,Liverpool,2,1,away,win,3,15
440,2024-10-05,Liverpool,1,0,away,win,3,18
78,2024-10-20,Liverpool,2,1,home,win,3,21
469,2024-10-27,Liverpool,2,2,away,draw,1,22
93,2024-11-02,Liverpool,2,1,home,win,3,25


In [16]:
team_matches.to_csv('team_matches_clean.csv', index=False)
standings.to_csv('standings_clean.csv', index=True)

print("Exported both files.")

Exported both files.


## Conclusions

Analysis of all 380 matches from the 2024/25 Premier League season, reshaped to a team-level
dataset (760 rows, one per team per match), surfaced the following:

**1. Liverpool won the title comfortably.** They finished on 84 points (25W-9D-4L) with a +45
goal difference, 10 points clear of second-placed Arsenal — the running points total shows them
pulling clear from the midpoint of the season onward.

**2. Home advantage is real on average, but far from universal.** Home teams won 155 matches to
the away side's 132 (40.8% vs 34.7%). Yet a quarter of the league gained no benefit from home
form: five teams finished level or better away than at home.

**3. Home advantage varied enormously by team.** Aston Villa were the strongest at home relative
to away (+14 points), followed by Chelsea and Manchester City (+13 each). At the other extreme,
Ipswich were actually *better away* (-8) — their poor home form (7 points from 19 games) was a
notable factor in their relegation.

**4. Defensive collapse defined the relegation zone.** Southampton (12 points, -60 goal difference),
Ipswich, and Leicester were relegated, each conceding 80+ goals across the season.

---

*Data reshaped and analysed in pandas; results validated against the official final standings.
Visualisations built in Tableau Public.*